In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Krylov Subspace Diagonalization — Chemistry Simulations
$\renewcommand{\ket}[1]{|#1\rangle}$

---

**What You Will Do:**
* Understand the theory behind the quantum Krylov subspace diagonalization (Krylov QSD) method and how it bridges NISQ and fault-tolerant quantum chemistry approaches
* Implement the Hadamard test-based quantum Krylov algorithm in CUDA-Q and batch its observables to reduce simulator launches
* Improve Krylov QSD accuracy by exploring additional first-order Trotter–Suzuki steps and multi-reference state construction
* Parallelize matrix element computation across multiple simulated QPUs using CUDA-Q's MQPU backend

**Prerequisites:**
* Python and Jupyter familiarity
* Basic knowledge of quantum computing (qubits, gates, measurement, quantum circuits)
* Familiarity with quantum chemistry concepts (Hamiltonians, Slater determinants, ground state energy) — the [VQE and GQE](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/vqe_and_gqe.ipynb) notebook provides a suitable introduction
* Experience with CUDA-Q kernels, `cudaq.observe`, and `cudaq.get_state` — the [Quick Start to Quantum Computing with CUDA-Q](https://github.com/NVIDIA/cuda-q-academic/tree/main/quick-start-to-quantum) series provides a suitable introduction

**Key Terminology:**
* Krylov subspace
* Quantum subspace diagonalization (QSD)
* Hadamard test
* Trotter–Suzuki approximation

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a quantum kernel function
* [`cudaq.qvector`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qvector) — allocates a register of qubits
* [`cudaq.observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) — computes expectation value of a spin operator
* [`cudaq.observe_async`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe_async) — asynchronous expectation value (multi-GPU)
* [`cudaq.get_state`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.get_state) — returns the statevector
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects simulation or hardware backend
* [`cudaq.pauli_word`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.pauli_word) — represents a fixed-width Pauli word
* [`cudaq.SpinOperator`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.SpinOperator) — Pauli spin operator (Hamiltonian)
* [`exp_pauli`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html) — applies $e^{i\theta P}$ for a Pauli word $P$

**Solutions:** [`solutions/krylov_subspace_diagonalization_solutions.ipynb`](solutions/krylov_subspace_diagonalization_solutions.ipynb)

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 12px 15px 12px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900;">&#9889; GPU Required:</span>** Parts of this notebook require a GPU.

</div>

In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.

#!pip install cudaq -q
#!pip install pyscf -q
#
#!wget -q https://github.com/nvidia/cuda-q-academic/archive/refs/heads/main.zip
#!unzip -q main.zip
#!mv cuda-q-academic-main/chemistry-simulations/aux_files ./aux_files
#!mv cuda-q-academic-main/chemistry-simulations/Images ./Images

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
import sys
import time
from pathlib import Path

lesson_dir = Path.cwd()
if lesson_dir.name == 'solutions':
    lesson_dir = lesson_dir.parent
sys.path.append(str(lesson_dir))

import numpy as np
import scipy

import cudaq

from aux_files.krylov.qchem.classical_pyscf import get_mol_hamiltonian
from aux_files.krylov.qchem.hamiltonian import jordan_wigner_fermion
from aux_files.krylov.qchem.eigenvaluesolver import eigen

# Single-node, single GPU
cudaq.set_target("nvidia", option='fp64')

---

## 1. Quantum Subspace Diagonalization

Quantum computing methods for quantum chemistry lie on a spectrum from NISQ methods to methods suited for large-scale fault tolerant quantum computers. The problem that we'll be considering in this notebook is to find the ground state energy of a Hamiltonian.  Mathematically, this is equivalent to finding the lowest eigenvalue of a given matrix.

The prototypical NISQ algorithm is the Variational Quantum Eigensolver (VQE), which computes ground state energies by classically optimizing a parameterized circuit and computing expectation values for the cost function. These circuits can be relatively shallow, but electronic-structure Hamiltonians commonly contain $O(n^4)$ Pauli terms, each of which contributes to the measurement cost. In addition, some parameterized circuits develop very small optimization gradients—known as barren plateaus—which can impede convergence. Even when convergence is possible, the number of iterations required may become intractably large.

On the other side of the spectrum, quantum phase estimation (QPE) is a hallmark fault-tolerant algorithm for quantum chemistry. Given a good initial state, QPE offers systematically improvable energy precision, but high precision requires long coherent controlled time evolutions. Large chemistry applications are therefore generally expected to rely on quantum error correction ([learn more in our QEC 101 course](https://github.com/NVIDIA/cuda-q-academic/tree/main/qec101)), which introduces substantial physical-qubit overhead.

Quantum Krylov methods are a bridge between the two and balance tradeoffs to potentially result in useful applications on devices that are more near term. The main idea is to reduce the size of the matrix to a smaller one and compute the eigenvalues of this smaller matrix.  If chosen well, the lowest eigenvalues of the smaller matrix and the larger one agree. The Krylov approach is a subset of broader **quantum subspace diagonalization (QSD)** methods which try to diagonalize the Hamiltonian in a general non-orthogonal basis. In other words, given an initial state that is close to the ground state, the Krylov method will help identify a good subspace of the Hamiltonian to diagonalize such that the best possible estimates of the extreme eigenvalues (i.e., the ground state) are produced.

Such an approach requires circuits of moderate depth and uses them to populate matrix elements of a Hamiltonian subspace which is then classically diagonalized.   Krylov methods still suffer from the O($n^4$) Hamiltonian scaling as the full Hamiltonian must be measured to compute subspace matrix elements.  Increasing the subspace dimension can systematically improve the ground-state estimate, although the rate of convergence depends on the reference states and time points used.

The widget below will provide some intuition for subspace diagonalization and why Krylov methods work.  Given the 8x8 matrix below, move the slider to select a subspace dimension and look at the computed eigenvalues.  Notice the ground state energy converges to the exact ground state while the other eigenvalues remain rather inaccurate.  Subspace methods are quite good at estimating the extreme eigenvalues (highest or lowest) which is favorable as the lowest eigenvalue (the ground state energy) is usually the quantity of interest.

Click [this link](https://nvidia.github.io/cuda-q-academic/chemistry-simulations/Images/krylov.html) to access the widget

To get a sense for why a good initial state is important (more on that later), choose which corner the subspace originates from in the widget. Notice, the approximate eigenvalues are much worse when the subspace is constructed from certain corners which happen to capture less information about the eigen spectrum of the matrix.  

Similarly, in the quantum Krylov methods, a good approximation of the ground state will help construct a better subspace for estimating the ground state energy.

---

## 2. The Quantum Krylov Method

This section will walk through an implementation of a quantum Krylov method based on [A Multireference Quantum Krylov Algorithm for Strongly
Correlated Electrons](https://arxiv.org/pdf/1911.05163). The general workflow follows the figure below inspired by [Quantum Krylov subspace algorithms
for ground and excited state energy estimation](https://www.osti.gov/servlets/purl/1962060).    The first step is to build a subspace for which a generalized eigenvalue problem will be solved.  Then, the matrices for the eigenvalue problem are populated with expectation values from quantum circuit evaluations using the **Hadamard test**, and finally, the eigenvalues of the subspace are determined classically. The subspace can then be gradually increased in size and the process repeated until the result is sufficiently accurate.

<img src="Images/krylov/krylov_approach.png" alt="Flowchart of the Krylov QSD workflow: start with reference states, apply time evolution unitaries to build the Krylov subspace, evaluate the overlap matrix S and Hamiltonian matrix H via Hadamard test circuits, then classically diagonalize the generalized eigenvalue problem to obtain the ground state energy estimate." title="Krylov Workflow" width="900">



The first step is to select a set of reference states from which the subspace is constructed. A benefit for this approach is that it is multireference. A set of $d$ reference states $\{\Phi_0, \ldots, \Phi_{d-1}\}$ is defined where each state is a linear combination of Slater determinants: 

$$ \ket{\Phi_I}  =  \sum_{\mu} d_{\mu I}\ket{\phi_{\mu}}. $$



From this, a non-orthogonal **Krylov subspace** $\mathcal{K} = \{\psi_{0}, \ldots, \psi_{N-1}\}$ is constructed by applying $s$ time-evolution operators to each of the $d$ reference states, resulting in $N=d\times s$ elements in the Krylov space, where
$$ \ket{\psi_{\alpha}} \equiv \ket{\psi_I^{(n)}} = \hat{U}_n\ket{\Phi_I}, \qquad I=0,\ldots,d-1,\; n=0,\ldots,s-1. $$

A linear combination of these basis states defines the Krylov approximation to the full configuration interaction (FCI) wavefunction.



$$ \ket{\Psi} = \sum_{\alpha} c_{\alpha}\ket{\psi_{\alpha}} = \sum_{I=0}^{d-1} \sum_{n=0}^{s-1} c_I^{(n)}\hat{U}_n\ket{\Phi_I}.  $$

The unitary operations are real-time evolutions. This notebook defines $U_n=e^{iHt_n}$ to match CUDA-Q's `exp_pauli` convention; using the opposite sign corresponds to choosing negative time points. These related states form the non-orthogonal Krylov basis.


The energy of this state can be obtained by solving the generalized eigenvalue problem using the state defined above to compute the matrix elements of $H$ and $S$. 

$$
\boldsymbol{Hc}=\boldsymbol{Sc}E
$$


These elements are obtained using the Hadamard test, which uses phase kickback to estimate an expectation value of the time evolution operators.
If $H = \sum_l c_lP_l$, 
the elements of the overlap are

$$
S_{\alpha \beta} = \langle\psi_{\alpha}|\psi_{\beta}\rangle = \langle\Phi_I|\hat{U}_m^{\dagger}\hat{U}_n|\Phi_J\rangle
$$

and the elements for the Hamiltonian matrix are

$$
H_{\alpha \beta} = \langle\psi_{\alpha}|\hat{H}|\psi_{\beta}\rangle = \langle\Phi_I|\hat{U}_m^{\dagger}\hat{H}\hat{U}_n|\Phi_J\rangle = \sum_l c_l \langle\Phi_I|\hat{U}_m^{\dagger}\hat{P_l}\hat{U}_n|\Phi_J\rangle
$$



The matrix elements for $S$ and $H$ are computed with the Hadamard test using the circuit shown below. In the case of the overlap matrix $S$, the Pauli word is the identity, so the $P_l$ drops out.

<img src="Images/krylov/krylov_hadamard_circuit.png" alt="Quantum circuit diagram for the Hadamard test used to compute matrix elements. An ancilla qubit is placed in superposition by a Hadamard gate, then used to control the application of time evolution kernel U_m (dagger) on the data register, followed by an optional Pauli operator P_l and time evolution kernel U_n. A final Hadamard and measurement on the ancilla yields the real or imaginary part of the matrix element." title="Krylov Hadamard Circuit" width="900">


The $2\sigma_+$ term refers to measurement of the expectation value of this circuit with the $X+iY$ operator.

After the controlled time evolutions, the circuit prepares

$$
|\Omega_{\alpha\beta}\rangle
=\frac{|0\rangle|\psi_\alpha\rangle+|1\rangle|\psi_\beta\rangle}{\sqrt{2}}.
$$

For any system Pauli word $P_l$, this state satisfies

$$
\langle X_a\otimes P_l\rangle+i\langle Y_a\otimes P_l\rangle
=\langle\psi_\alpha|P_l|\psi_\beta\rangle.
$$

The implementation therefore moves $P_l$ from the controlled circuit branch into the measured observable. This leaves the matrix element unchanged while allowing CUDA-Q to evaluate all Pauli terms from one prepared simulator state.

Once the $H$ and $S$ matrices are constructed, the diagonalization is performed classically to produce an estimate for the ground state in question.


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 1:</span>**

Before implementing the quantum Krylov method, find the ground state energy of the linear $H_4$ molecule and compute the FCI energy by direct matrix diagonalization. This will provide a reference to compare to your Krylov energies. Note that the `spin_ham_matrix = hamiltonian.to_matrix()` line saves the Hamiltonian as a NumPy array. How large is the full Hamiltonian matrix?

</div>

In [ ]:
# EXERCISE 1
#geometry = 'Li 0.3925 0.0 0.0; H -1.1774 0.0 0.0'
#geometry = 'H 0.0 0.0 0.0; H 0.0 0.0 0.7474'
geometry = 'H 0.0 0.0 1.5; H 0.0 0.0 3.0; H 0.0 0.0 4.5; H 0.0 0.0 6.0'
molecular_data = get_mol_hamiltonian(xyz=geometry, spin=0, charge=0, basis='sto3g', ccsd=True, verbose=True)

obi = molecular_data[0]
tbi = molecular_data[1]
e_nn = molecular_data[2]
nelectrons = molecular_data[3]
norbitals = molecular_data[4]

qubits_num = 2 * norbitals

hamiltonian = jordan_wigner_fermion(obi, tbi, e_nn, tolerance = 1e-12)

spin_ham_matrix = hamiltonian.to_matrix()

print("H is a", len(spin_ham_matrix), "by", len(spin_ham_matrix) , "matrix")

##TODO## Compute the FCI eigensystem with np.linalg.eigh
e, c = None, None  ##TODO##

# Find the ground state energy and the corresponding eigenvector
print('Ground state energy (classical simulation)= ', np.min(e), ', index= ',
      np.argmin(e))
min_indices = np.argsort(e)[:5]

# Eigenvector can be used to initialize the qubits
vec = c[:, min_indices[0]]

---

## 3. Implementing a Quantum Krylov Method

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 2:</span>**

Most of `qfd_kernel` is provided. Complete its final three operations: apply the $\alpha$-branch evolution, flip the ancilla, and apply the $\beta$-branch evolution.

</div>

#### Building controlled time evolution

CUDA-Q's `exp_pauli(theta, register, P)` implements $e^{i\theta P}$. To apply this exponential only when an ancilla is $|1\rangle$, use the projector

$$
|1\rangle\!\langle 1| = \frac{I-Z_a}{2}.
$$

For a system Pauli word $P$, this gives the exact identity

$$
C_1(e^{i\theta P})
=e^{i\theta |1\rangle\!\langle 1|\otimes P}
=e^{i\theta I_a\otimes P/2}\,e^{-i\theta Z_a\otimes P/2}.
$$

The two factors commute, so `controlled_time_evolution` implements them with two ordinary `exp_pauli` calls. The ancilla is qubit 0 of the combined register. Prefixing a system word $P$ with `I` produces $I_a\otimes P$; prefixing it with `Z` produces $Z_a\otimes P$. CUDA-Q handles the gate-level synthesis of both exponentials.

To prepare two different Slater determinants, start the system register in $|\Phi_\beta\rangle$. Controlled $X$ gates on the symmetric difference of the occupied-orbital lists turn only the ancilla-$|1\rangle$ branch into $|\Phi_\alpha\rangle$. After applying the two time evolutions, the Hadamard-test state has the form

$$
|\Omega_{\alpha\beta}\rangle
=\frac{|0\rangle|\psi_\alpha\rangle+|1\rangle|\psi_\beta\rangle}{\sqrt{2}}.
$$


In [ ]:
@cudaq.kernel
def controlled_time_evolution(
        register: cudaq.qview, dt: float, coefficients: list[complex],
        identity_words: list[cudaq.pauli_word],
        z_words: list[cudaq.pauli_word], trotter_steps: int):
    """Apply a controlled first-order Trotter approximation to exp(i H dt).

    For each Pauli word P, use |1><1| = (I - Z) / 2 to write its
    controlled exponential as two ordinary Pauli exponentials.
    """
    for _ in range(trotter_steps):
        for term in range(len(coefficients)):
            theta = dt * coefficients[term].real / trotter_steps
            exp_pauli(0.5 * theta, register, identity_words[term])
            exp_pauli(-0.5 * theta, register, z_words[term])


@cudaq.kernel
def qfd_kernel(dt_alpha: float, dt_beta: float,
               coefficients: list[complex],
               identity_words: list[cudaq.pauli_word],
               z_words: list[cudaq.pauli_word],
               state_beta: list[int], state_difference: list[int],
               n_qubits: int, trotter_steps: int):
    """Prepare the two time-evolved branches used by the Hadamard test."""
    # The ancilla is register[0]; system qubit j is register[j + 1].
    register = cudaq.qvector(n_qubits + 1)
    ancilla = register[0]

    # Prepare |Phi_beta> on both branches.
    for index in range(len(state_beta)):
        x(register[state_beta[index] + 1])

    h(ancilla)

    # On the |1> branch, toggle differing occupations to obtain |Phi_alpha>.
    for index in range(len(state_difference)):
        x.ctrl(ancilla, register[state_difference[index] + 1])

    ##TODO## Apply the alpha evolution, flip the ancilla, and apply the
    # beta evolution. This requires two controlled_time_evolution calls and
    # one x(ancilla) operation.


CUDA-Q kernels require explicit argument types. `prepare_hamiltonian_data` extracts the Hamiltonian coefficients and fixed-width system Pauli words. It also constructs the two lists used by `controlled_time_evolution`: each system word is prefixed with `I` or `Z` for the ancilla. For example, the system word `XXZI` becomes `IXXZI` and `ZXXZI` on the combined ancilla-plus-system register.

Run `prepare_hamiltonian_data` whenever the molecular Hamiltonian changes, and then rebuild the compound measurement operator from the system Pauli words. You only need to complete the two one-line extraction functions; the prefixed-word construction is provided.


In [ ]:
# Collect coefficients from a spin operator
def term_coefficients(ham: cudaq.SpinOperator) -> list[complex]:
    """Return the scalar coefficient of every Hamiltonian term."""
    ##TODO## Return one evaluated coefficient for each term in ham.


def term_words(ham: cudaq.SpinOperator, n_qubits: int) -> list[str]:
    """Return every Hamiltonian Pauli word with a fixed register width."""
    ##TODO## Return one fixed-width Pauli word for each term in ham.


def prepare_hamiltonian_data(ham: cudaq.SpinOperator, n_qubits: int):
    """Prepare system words and the two ancilla-prefixed word lists."""
    coefficients = term_coefficients(ham)
    pauli_words = term_words(ham, n_qubits)
    identity_words = [
        cudaq.pauli_word('I' + word) for word in pauli_words
    ]
    z_words = [
        cudaq.pauli_word('Z' + word) for word in pauli_words
    ]
    return coefficients, pauli_words, identity_words, z_words


coefficient, pauli_string, identity_words, z_words = prepare_hamiltonian_data(
    hamiltonian, qubits_num)

print(coefficient[0:10])
print(pauli_string[0:10])
print(str(identity_words[0]), str(z_words[0]))


#### Computing the matrix elements

The same prepared Hadamard-test state contains the information needed for both projected matrices. For an inserted Pauli word $P_l$,

$$
\langle X_a\otimes P_l\rangle + i\langle Y_a\otimes P_l\rangle
= \langle\psi_\alpha|P_l|\psi_\beta\rangle.
$$

For $S$, set $P_l=I$. For $H=\sum_l c_lP_l$, multiply each result by $c_l$ and sum. `measurement_data` builds one compound `SpinOperator` containing $X_a$, $Y_a$, and every non-identity $X_a\otimes P_l$ and $Y_a\otimes P_l$. `matrix_elements` extracts both $H_{mn}$ and $S_{mn}$ from the resulting `ObserveResult`.

On an analytic simulator, one `cudaq.observe` call can reuse a single prepared state for this compound operator. On shot-based hardware, CUDA-Q must still execute the measurement circuits required by the noncommuting Pauli terms, so batching reduces host launches but not the underlying measurement statistics.

The loops and result-extraction structure are provided. Complete only the `X + word` and `Y + word` observable construction and the coefficient-weighted Hamiltonian sum.


In [ ]:
# Ancilla observables for the real and imaginary Hadamard tests.
x_0 = cudaq.spin.x(0)
y_0 = cudaq.spin.y(0)


def measurement_data(pauli_words: list[str], n_qubits: int):
    """Build one compound operator for S and H matrix elements."""
    measurement_operator = x_0 + y_0
    x_terms = []
    y_terms = []
    identity = 'I' * n_qubits

    for word in pauli_words:
        ##TODO## Prefix each system word with the ancilla X and Y.
        x_term = None
        y_term = None
        x_terms.append(x_term)
        y_terms.append(y_term)

        # X/Y tensor identity are already present as x_0 and y_0.
        if word != identity:
            measurement_operator += x_term + y_term

    return measurement_operator, x_terms, y_terms


def matrix_elements(result: cudaq.ObserveResult) -> tuple[complex, complex]:
    """Extract one Hamiltonian and overlap element from an ObserveResult."""
    overlap_element = result.expectation(x_0) + 1j * result.expectation(y_0)
    ##TODO## Replace 0j with the coefficient-weighted sum of
    # <X_a tensor P_l> + i<Y_a tensor P_l> over all Hamiltonian terms.
    hamiltonian_element = 0j
    return hamiltonian_element, overlap_element


measurement_operator, x_terms, y_terms = measurement_data(
    pauli_string, qubits_num)


Now populate $H$ and $S$ together. The traversal and Hermitian filling are provided; replace the single placeholder in `cudaq.observe` with the compound `measurement_operator`. Only the upper triangle is evaluated because both matrices are Hermitian. Each upper-triangle pair requires one compound-observable call instead of two calls for $S$ plus two calls for every Hamiltonian term. The corresponding lower-triangle entries are filled by complex conjugation.


In [ ]:
def populate_matrices(dt, n_steps, ref_states, trotter_steps=1):
    """Compute the projected Hamiltonian H and overlap S together."""
    dimension = n_steps * len(ref_states)
    ham_matrix = np.zeros((dimension, dimension), dtype=complex)
    wf_overlap = np.zeros((dimension, dimension), dtype=complex)

    # Order the basis by time point, then by reference state.
    dt_s = [step * dt for step in range(n_steps) for _ in ref_states]
    states = ref_states * n_steps

    for m in range(dimension):
        for n in range(m, dimension):
            # Starting from state_n, these controlled X gates prepare state_m
            # on the other ancilla branch.
            state_difference = sorted(set(states[m]) ^ set(states[n]))

            result = cudaq.observe(
                qfd_kernel, None, dt_s[m], dt_s[n],  ##TODO## Use the compound observable.
                coefficient, identity_words, z_words, states[n], state_difference,
                qubits_num, trotter_steps)

            h_element, s_element = matrix_elements(result)
            ham_matrix[m, n] = h_element
            wf_overlap[m, n] = s_element
            if n != m:
                ham_matrix[n, m] = np.conj(h_element)
                wf_overlap[n, m] = np.conj(s_element)

    return ham_matrix, wf_overlap


#### Solving the generalized eigenvalue problem

The quantum calculations produce a projected Hamiltonian $H$ and a non-orthogonal overlap matrix $S$. The coefficients satisfy

$$
H C = S C E.
$$

The imported `eigen` function first diagonalizes the Hermitian overlap matrix,

$$
S = U\Sigma U^{\dagger}.
$$

Very small eigenvalues of $S$ are discarded to remove nearly linearly dependent Krylov vectors. From the retained eigenpairs it constructs

$$
X = U_{\mathrm{keep}}\Sigma_{\mathrm{keep}}^{-1/2},
\qquad X^{\dagger} S X = I.
$$

It then solves the ordinary Hermitian eigenvalue problem

$$
(X^{\dagger} H X) C' = C' E
$$

and transforms the eigenvectors back with $C=XC'$. The **condition number** $\kappa(S)$ is the ratio of the largest to the smallest retained positive eigenvalue. A very large value signals near-linear dependence and possible numerical instability. Set `verbose=True` when calling `eigen` to print the conditioning diagnostics.


Now, compute the ground state energy of $H_4$ using the Hartree–Fock state as the only reference, evolving 4 and then 8 time steps. (The first example is provided for running this workflow.)  Then recompute using two reference states, HF and a determinant formed by promoting an electron from the highest occupied orbital to the next available orbital and evolve each for 2 and then 4 time steps. 

How do your results look? Is a larger $N=s*d$ better?  Are better results obtained from increasing $d$ or $s$?  Are they accurate to within chemical accuracy (about 1.6 millihartree)?  Note, you must use FP64 precision with the `nvidia` simulator; otherwise, your result will suffer from numerical inaccuracies.

In [ ]:
timesteps = 4
ref_states = [[0,1,2,3]]
dt= 1.0


H_final, S_final = populate_matrices(dt, timesteps, ref_states)

eigen_value, eigen_vect = eigen(H_final, S_final)
print('Energy from QFD s = 4, d = 1:')
print(np.min(eigen_value))



##TODO## Use the Hartree Fock state evolved 8 time steps
timesteps = None  ##TODO##
ref_states = None  ##TODO##
dt= 1.0


H_final, S_final = populate_matrices(dt, timesteps, ref_states)

eigen_value, eigen_vect = eigen(H_final, S_final)
print('Energy from QFD s = 8, d = 1:')
print(np.min(eigen_value))

##TODO## Use the Hartree Fock state and an excited state Slater Determinant, both evolved 2 steps.
timesteps = None  ##TODO##
ref_states = None  ##TODO##
dt= 0.001
H_final, S_final = populate_matrices(dt, timesteps, ref_states)


eigen_value, eigen_vect = eigen(H_final, S_final)
print('Energy from QFD s = 2, d = 2:')
print(np.min(eigen_value))

##TODO## Use the Hartree Fock state and an excited state Slater Determinant, both evolved 4 steps.
timesteps = None  ##TODO##
ref_states = None  ##TODO##
dt= 0.001
H_final, S_final = populate_matrices(dt, timesteps, ref_states)


eigen_value, eigen_vect = eigen(H_final, S_final)
print('Energy from QFD s = 4, d = 2:')
print(np.min(eigen_value))


print('Exact Ground State Energy:')
print(np.min(e))

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 3:</span>**

For STO-3G LiH, suppose the Hamiltonian contains $L=631$ Pauli terms and the Krylov dimension is $N=d\times s=6$. There are $N(N+1)/2$ upper-triangle matrix pairs. Compute:

1. The number of host-side calls made by the batched implementation: one call per matrix pair.
2. The number made by the original term-by-term implementation: two calls for $S$ and two calls for each of the $L$ Hamiltonian terms, per matrix pair.

For scale, compare the old host-call count with the $631{,}000$ term evaluations in a hypothetical 1000-iteration VQE calculation, and compare the six-dimensional Krylov subspace with the $2^{12}$-dimensional LiH Hilbert space. These host-call counts are not physical circuit counts: on shot-based hardware, measurement grouping and shot allocation determine the actual circuit workload.

</div>


In [ ]:
# EXERCISE 3
def krylov_host_calls(dimension, ham_terms):
    upper_triangle = dimension * (dimension + 1) // 2
    batched_calls = upper_triangle
    term_by_term_calls = None  ##TODO## Add the original per-pair X/Y calls.
    return batched_calls, term_by_term_calls


batched_calls, old_calls = krylov_host_calls(6, 631)
print('Batched host calls:', batched_calls)
print('Original term-by-term host calls:', old_calls)
print('Term evaluations for 1000 VQE iterations:', 1000 * 631)
print('Full Hilbert-space dimension / Krylov dimension:', 2**12 / 6)


The batched implementation reduces the number of Python-to-backend submissions dramatically while preserving the same matrix elements. That comparison should not be interpreted as an equally large reduction in physical measurements: a shot-based device must still estimate the required Pauli observables to the desired statistical precision. The Krylov method also avoids a variational optimization loop, but its total cost still grows with the number of Hamiltonian terms and Krylov matrix pairs.


---

## 4. Selecting an Improved Krylov Basis

As you saw from your initial implementation, the quality of the Krylov QSD result can depend heavily on the reference states used and how many time evolution steps are selected. Examine the table below from the paper. Notice how larger subspaces improve (lower) the ground state energy prediction for the linear $H_6$ molecule (STO-6G). 

The table tells a second story. The columns labeled $\kappa$ denote the **condition number** of the overlap matrix $S$: the ratio of its largest to smallest retained positive eigenvalue. If this number is very large (over $10^{12}$), the solution can become numerically unstable and produce inaccurate or even non-variational predictions. This tends to occur when vectors in the Krylov subspace are too similar.

The table has two columns, one where a single reference (SR) Hartree Fock state is evolved for $N$ time steps, and another where multiple reference vectors $d = N/4$ are evolved for four time steps each.  Notice how the MR condition numbers are small and remain below $10^7$ while the SR cases are huge.  Though potentially OK for a small system, this is highly problematic for larger molecules and indicates the value of using multiple reference vectors over a single reference evolved for more time steps.  


| Subspace Size | E(SR) | $$\kappa(\text{SR})$$ | E(MR) | $$\kappa(\text{MR})$$ |
|---:|-------------:|----------------------:|--------------:|---------------------------:|
| 4  | −3.015510   | $$3.29\times10^{5}$$  | −3.015510 | $$3.29\times10^{5}$$ |
| 8  | −3.019768   | $$3.60\times10^{11}$$ | −3.019301 | $$4.86\times10^{5}$$ |
| 12 | −3.020172   | $$1.61\times10^{17}$$ | −3.019696 | $$9.39\times10^{5}$$ |
| 16 | −3.020192   | $$3.19\times10^{17}$$ | −3.019835 | $$5.68\times10^{6}$$ |
| 20 | −3.020198   | $$3.86\times10^{17}$$ | −3.019929 | $$6.23\times10^{6}$$ |


A natural question is what other choices can improve the construction of the Krylov subspace?  The rest of this section will explore two more ways. 



<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 4:</span>**

One way to improve the subspace is to choose an appropriate time evolution step. Use your code to test the following time steps $\{10, 1, ... 0.00001\}$ with the 6-31G Hartree Fock state of $H_2$ (so it runs faster) as a reference time-evolved for three Krylov time points. Comment on the ground state energy predictions. Is there a systematic trend? What might be an explanation for what makes a good or poor $dt$ selection?

</div>

In [ ]:
# EXERCISE 4
geometry = 'H 0.0 0.0 0.0; H 0.0 0.0 0.7474'
molecular_data = get_mol_hamiltonian(xyz=geometry, spin=0, charge=0, basis='631g', ccsd=True, verbose=False)

obi = molecular_data[0]
tbi = molecular_data[1]
e_nn = molecular_data[2]
nelectrons = molecular_data[3]
norbitals = molecular_data[4]

qubits_num = 2 * norbitals

hamiltonian = jordan_wigner_fermion(obi, tbi, e_nn, tolerance = 1e-12)

spin_ham_matrix = hamiltonian.to_matrix()

e, c = np.linalg.eigh(spin_ham_matrix)

# Build the lists of coefficients and Pauli Words from the H2 Hamiltonian
coefficient, pauli_string, identity_words, z_words = prepare_hamiltonian_data(
    hamiltonian, qubits_num)
measurement_operator, x_terms, y_terms = measurement_data(
    pauli_string, qubits_num)


timesteps = 3
ref_states = [[0,1]]

for dt in [10, 1.0, 0.1, 0.01, 0.001, 0.0001, 0.00001]:
    
    H_final, S_final = populate_matrices(dt, timesteps, ref_states)

    eigen_value, eigen_vect = eigen(H_final, S_final)
    
    print("The time step dt=", dt, "results in E=", np.min(eigen_value))


print('Exact Ground State Energy:')
print(np.min(e))

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 5:</span>**

Another way to improve the basis selection has to do with how the time evolution operator is approximated. Consider the Hamiltonian $H = \sum c_lP_l$ where $c_l$ are the coefficients and $P_l$ are the Pauli words. The `controlled_time_evolution` kernel applies each controlled Pauli exponential using the two commuting `exp_pauli` factors derived above. This is necessary because noncommuting terms must be ordered using the **Trotter–Suzuki approximation** shown below.

$ e^{i(c_0P_0 + c_1P_1 + \cdots c_lP_l)t} \approx (e^{ic_0P_0t/k}e^{ic_1P_1t/k}\cdots e^{ic_lP_lt/k})^k, $

where the approximation converges to the exact result as the number of repeated steps $k$ increases. The `trotter_steps` argument stores this value of $k$, the number of repeated first-order product-formula steps. Use it to run $H_4$ from the Hartree–Fock reference with three Krylov time points and $k = \{1,2,3,4,5\}$. Time each calculation. Do larger values of $k$ improve the energy, and how does the runtime change?

</div>

In [ ]:
# EXERCISE 5
geometry = 'H 0.0 0.0 1.5; H 0.0 0.0 3.0; H 0.0 0.0 4.5; H 0.0 0.0 6.0'
molecular_data = get_mol_hamiltonian(xyz=geometry, spin=0, charge=0, basis='sto3g', ccsd=True, verbose=False)

obi = molecular_data[0]
tbi = molecular_data[1]
e_nn = molecular_data[2]
nelectrons = molecular_data[3]
norbitals = molecular_data[4]

qubits_num = 2 * norbitals

hamiltonian = jordan_wigner_fermion(obi, tbi, e_nn, tolerance = 1e-12)

spin_ham_matrix = hamiltonian.to_matrix()

e, c = np.linalg.eigh(spin_ham_matrix)

coefficient, pauli_string, identity_words, z_words = prepare_hamiltonian_data(
    hamiltonian, qubits_num)
measurement_operator, x_terms, y_terms = measurement_data(
    pauli_string, qubits_num)


timesteps = 3
ref_states = [[0,1,2,3]]
dt = 0.1
for k in [1,2,3,4,5]:
    start = time.time()
    H_final, S_final = populate_matrices(
        dt, timesteps, ref_states, k)

    eigen_value, eigen_vect = eigen(H_final, S_final)
    end = time.time()
    print("The number of Trotter steps k=", k, "results in E=", np.min(eigen_value), "with time:", end - start)


print('Exact Ground State Energy:')
print(np.min(e))

Increasing the number of Trotter steps can improve the result, but it also has a cost. The time-evolution gate count grows approximately linearly with $k$, so larger values require more gates and longer circuit simulation times. In [A Multireference Quantum Krylov Algorithm for Strongly Correlated Electrons](https://arxiv.org/pdf/1911.05163), the authors report $BeH_2$ energy errors for increasingly fine Trotter discretizations labeled by $m$. The table shows the same accuracy-versus-cost trend: finer discretization generally lowers the product-formula error.


<img src="Images/krylov/trotterordererror.png" alt="A table from a published study showing ground-state energy errors for BeH₂ at several bond distances and Trotter discretizations. Finer discretizations generally reduce the energy error, illustrating the trade-off between circuit depth and accuracy." title="Trotter discretization error table" width="700">


---

## 5. Parallel Krylov Method

Batching the Pauli observables removes most repeated host launches on an analytic simulator, but a larger Krylov space still contains many independent upper-triangle matrix elements. Those matrix-element jobs are naturally parallel: each prepares a different pair of Krylov basis vectors and returns both one $H$ entry and one $S$ entry.

CUDA-Q's MQPU backend exposes multiple QPUs—or multiple GPUs used as simulated QPUs—through `qpu_id`. The next exercise submits one compound-observable `observe_async` call per upper-triangle pair and assigns those jobs to available QPUs in round-robin order. On a one-GPU system the code remains correct but should not be expected to speed up; the benefit appears when `cudaq.get_target().num_qpus()` is greater than one and the jobs are sufficiently expensive.

The logical measurement cost discussed in Exercise 3 still applies on shot-based hardware. Here, parallelism changes when independent work is performed; it does not remove that work.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 6:</span>**

The submission and recovery loops are provided. Complete the round-robin `qpu_id` expression for each asynchronous job, then compare its runtime and energy with `populate_matrices`.

</div>


In [ ]:
# EXERCISE 6
cudaq.set_target('nvidia', option='mqpu,fp64')
num_qpus = cudaq.get_target().num_qpus()
print('Available simulated QPUs:', num_qpus)


def populate_matrices_async(dt, n_steps, ref_states, trotter_steps=1):
    """Distribute independent upper-triangle matrix elements across QPUs."""
    dimension = n_steps * len(ref_states)
    ham_matrix = np.zeros((dimension, dimension), dtype=complex)
    wf_overlap = np.zeros((dimension, dimension), dtype=complex)
    # Match the time-major ordering used by populate_matrices.
    dt_s = [step * dt for step in range(n_steps) for _ in ref_states]
    states = ref_states * n_steps
    jobs = []

    for m in range(dimension):
        for n in range(m, dimension):
            state_difference = sorted(set(states[m]) ^ set(states[n]))
            future = cudaq.observe_async(
                qfd_kernel, measurement_operator, dt_s[m], dt_s[n],
                coefficient, identity_words, z_words, states[n], state_difference,
                qubits_num, trotter_steps, qpu_id=None)  ##TODO## Use len(jobs) and num_qpus for round-robin assignment.
            jobs.append((m, n, future))

    for m, n, future in jobs:
        h_element, s_element = matrix_elements(future.get())
        ham_matrix[m, n] = h_element
        wf_overlap[m, n] = s_element
        if n != m:
            ham_matrix[n, m] = np.conj(h_element)
            wf_overlap[n, m] = np.conj(s_element)

    return ham_matrix, wf_overlap


timesteps = 3
ref_states = [[0, 1, 2, 3]]
dt = 0.1
k = 3

start = time.time()
H_serial, S_serial = populate_matrices(dt, timesteps, ref_states, k)
serial_time = time.time() - start

start = time.time()
H_async, S_async = populate_matrices_async(dt, timesteps, ref_states, k)
async_time = time.time() - start

print('Serial energy:', np.min(eigen(H_serial, S_serial)[0]),
      'time:', serial_time)
print('Async energy:', np.min(eigen(H_async, S_async)[0]),
      'time:', async_time)


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 7:</span>**

The maximum-error and speedup calculations are provided. Add two `np.allclose` assertions to verify that the serial and asynchronous $H$ and $S$ matrices agree within `atol=1e-10`. Explain why a one-QPU run may be slower and why two QPUs need not produce an exact $2\times$ speedup.

</div>


In [ ]:
# EXERCISE 7
h_error = np.max(np.abs(H_serial - H_async))
s_error = np.max(np.abs(S_serial - S_async))

print('Maximum |H_serial - H_async|:', h_error)
print('Maximum |S_serial - S_async|:', s_error)
print('Observed speedup:', serial_time / async_time)

##TODO## Add np.allclose assertions for H_serial/H_async and
# S_serial/S_async with atol=1e-10.


---

## Conclusion

The **Krylov QSD** approach is a powerful and versatile tool. This lab has introduced the underlying theory that allowed you to implement your own version and explore several ways to improve results, batch observable evaluation, and parallelize independent matrix elements.  

The hybrid nature of the Krylov approach is excellent for balancing the need for quantum computers to evaluate expectation values from large Hilbert spaces, but do so in a way that a subspace matrix can be constructed and solved classically.  A very promising aspect of Krylov methods is the fact that they can be combined with other quantum computing approaches. For example, it is perfectly valid to prepare the reference state and then run Krylov QSD to refine the prediction.  This further creates a robust hybrid workflow that builds upon other quantum techniques.  

For a challenge, if you have completed the [ADAPT-VQE](https://github.com/NVIDIA/cuda-q-academic/tree/main/chemistry-simulations) CUDA-Q Academic lab, see if you can combine these two methods into a single workflow.

**Related Notebooks:**
* [VQE and GQE](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/vqe_and_gqe.ipynb) — explores variational quantum eigensolver approaches for quantum chemistry
* [ADAPT-VQE](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/adapt_vqe.ipynb) — implements an adaptive variational algorithm that can be combined with Krylov methods
* [Quick Start to Quantum 04](https://github.com/NVIDIA/cuda-q-academic/blob/main/quick-start-to-quantum/04_quick_start_to_quantum.ipynb) — covers multi-GPU programming and the MQPU backend in CUDA-Q